In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler, TensorDataset
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_20newsgroups
from tqdm import tqdm
import numpy as np
import os
import random

In [2]:
# Ensure deterministic behavior
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


In [3]:
# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [4]:
# Load 20 Newsgroups dataset
newsgroups_train = fetch_20newsgroups(subset='train')


In [5]:
# Filter out empty texts
data = [(text, label) for text, label in zip(newsgroups_train.data, newsgroups_train.target) if text.strip() != ""]
texts, labels = zip(*data)

In [6]:
# Assign directly to train_texts and train_labels
train_texts = [text for text in newsgroups_train.data if text.strip() != ""]
train_labels = [label for text, label in zip(newsgroups_train.data, newsgroups_train.target) if text.strip() != ""]


In [7]:
# Tokenization
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(train_texts, padding=True, truncation=True, max_length=512)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [8]:
# Sanity checks
def check_encoding(encodings, labels):
    assert len(encodings['input_ids']) == len(labels), "Mismatch between input_ids and labels"
    assert all(len(ids) > 0 for ids in encodings['input_ids']), "Found empty input_id sequence"
    assert all(len(mask) > 0 for mask in encodings['attention_mask']), "Found empty attention_mask"

check_encoding(train_encodings, train_labels)


In [9]:
# Convert to tensors
train_input_ids = torch.tensor(train_encodings['input_ids'])
train_attention_mask = torch.tensor(train_encodings['attention_mask'])
train_labels = torch.tensor(train_labels, dtype=torch.long)



In [10]:
# Wrap in TensorDatasets
train_dataset = TensorDataset(train_input_ids, train_attention_mask, train_labels)


In [11]:
# Dataloaders
train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=8, num_workers=0)

In [12]:
# Load model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(set(newsgroups_train.target)))
model.to(device)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [13]:
# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
epochs = 3
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [14]:
# Training loop
def train():
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for batch in progress_bar:
            b_input_ids, b_input_mask, b_labels = [x.to(device) for x in batch]
            model.zero_grad()

            outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            progress_bar.set_postfix({"loss": loss.item()})

        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1} finished. Average Loss: {avg_loss:.4f}")


In [15]:
# Run training safely
if __name__ == "__main__":
    train()


Epoch 1 finished. Average Loss: 0.9845


Epoch 2 finished. Average Loss: 0.2818


Epoch 3 finished. Average Loss: 0.1323


In [19]:
# Save model and tokenizer
model_path = "./bert_model"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy to Google Drive
!mkdir -p /content/drive/MyDrive/bert_models
!cp -r ./bert_model /content/drive/MyDrive/bert_models/

print(" Model and tokenizer saved to: /content/drive/MyDrive/bert_models/bert_model")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Model and tokenizer saved to: /content/drive/MyDrive/bert_models/bert_model


In [20]:
from google.colab import drive
drive.mount('/content/drive')


from transformers import BertTokenizer, BertForSequenceClassification
import os

model_path = "/content/drive/MyDrive/bert_models/bert_model"

model = BertForSequenceClassification.from_pretrained(model_path, local_files_only=True)
tokenizer = BertTokenizer.from_pretrained(model_path, local_files_only=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
from sklearn.datasets import fetch_20newsgroups

test_data = fetch_20newsgroups(subset='test')
test_texts = test_data.data
test_labels = test_data.target




In [22]:
# Tokenize
test_encodings = tokenizer(test_texts, padding=True, truncation=True, max_length=512)
input_ids = torch.tensor(test_encodings['input_ids'])
attention_mask = torch.tensor(test_encodings['attention_mask'])
labels = torch.tensor(test_labels)


In [23]:
# Dataset and Dataloader
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler

test_dataset = TensorDataset(input_ids, attention_mask, labels)
test_dataloader = DataLoader(test_dataset, batch_size=8, sampler=SequentialSampler(test_dataset))


In [26]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
import torch

def evaluate_test_bert():
    model.eval()
    preds = []
    true_labels = []

    # Ensure the model is on the correct device
    model.to(device)

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Evaluating"):
            # Move batch tensors to the same device as the model
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            outputs = model(b_input_ids, attention_mask=b_input_mask)
            logits = outputs.logits

            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true_labels.extend(b_labels.cpu().numpy())

    acc = accuracy_score(true_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, preds, average='weighted')

    print(f"\n Evaluation Completed")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")

In [27]:
evaluate_test_bert()

Evaluating: 100%|██████████| 942/942 [06:30<00:00,  2.41it/s]


 Evaluation Completed
Test Accuracy: 0.8679
Precision: 0.8716, Recall: 0.8679, F1 Score: 0.8677


In [28]:
def predict_sample():
    sample_news = "mudiland had 4 prominent house on sale ."
    encoding = tokenizer(sample_news, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    model.eval()
    with torch.no_grad():
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        predicted_class = torch.argmax(output.logits, dim=1).item()

    print(f"Predicted class index: {predicted_class}")
    print(f"Predicted label name: {test_data.target_names[predicted_class]}")


In [29]:
predict_sample()

Predicted class index: 6
Predicted label name: misc.forsale
